# ES 335 genre classifier: CNN training on a Colab GPU

Runs `src/train_cnn.py` on split C with a GPU. Steps: clone the repo, install
requirements, give Kaggle credentials **for this session only**, download GTZAN,
compute the log-mels, train the config seeds, summarise, download the results.

Before running: *Runtime -> Change runtime type -> GPU*. The repo must contain the
CNN code on GitHub (push first). If the repo is private, clone with a GitHub token
instead of the plain URL.

Only split C **train and valid** are used; the test part stays locked in
`src/splits.py` until the `pre-test-freeze` tag exists.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

## 1. Clone and install

In [ ]:
%cd /content
!git clone https://github.com/yashvardhan-unlimited/ES-335-Music-genre-classifier.git repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Kaggle credentials (kept in memory only)

Upload your `kaggle.json` when prompted. The cell reads it, puts the username and
key into environment variables of this runtime, and deletes the uploaded copy at
once, so it is never stored in the repo or on disk after this cell. Nothing is
printed except the username.

In [ ]:
import json, os
from google.colab import files

os.chdir("/content")                      # upload lands outside the repo
uploaded = files.upload()                 # choose kaggle.json
name, raw = next(iter(uploaded.items()))
cred = json.loads(raw)
for n in uploaded:                        # files.upload() also writes the file: remove it
    if os.path.exists(n):
        os.remove(n)
os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = cred["username"], cred["key"]
del uploaded, raw, cred
assert not os.path.exists("/content/kaggle.json") and not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))
print("Kaggle user:", os.environ["KAGGLE_USERNAME"], "(credentials in memory only)")
os.chdir("/content/repo")

## 3. Download GTZAN and compute log-mels only

In [ ]:
!bash scripts/download_data.sh
!python -W ignore scripts/extract_features.py --mels-only
!ls features/mel | wc -l   # expect 999 (jazz.00054 is unreadable)

## 4. Train the CNN on split C (config seeds) and summarise

Each run early-stops on valid clip macro-F1 (patience 8, at most 60 epochs) and
logs the seconds per epoch.

In [ ]:
import yaml
seeds = yaml.safe_load(open("config.yaml"))["cnn"]["seeds"]
for s in seeds:
    !python -W ignore -m src.train_cnn --split C --seed {s} --epochs 60 --device cuda
!python -W ignore -m src.train_cnn --split C --summary

In [ ]:
from IPython.display import Image
Image("results/figures/cnn_curves_C.png")

## 5. Download the results

Zips the result JSONs, normalisation stats, curves figure and weights (no audio, no
mels, no credentials). Copy them into the local repo's `results/` and `models/`.

In [ ]:
!zip -q -r /content/cnn_results.zip results/val/C_s*_cnn*.json results/val/C_cnn_summary.json     results/mel_norm_C_s*.json results/figures/cnn_curves_C.png models/cnn_C_s*.pt
files.download("/content/cnn_results.zip")